# Матчинг товаров Мегамаркета: нейросеть-переранжировщик + LightGBM (стекинг)

Для каждой пары «предложение продавца (оффер) × кандидат из каталога» нужно решить, тот же это товар или нет.
Метрика — F1 по классу «матч» по всем парам. У оффера 5 кандидатов (изредка до 10), найденных поиском.

**Итог честной версии на Kaggle: private 0.7981, public 0.7981.**

| Шаг | F1 |
|---|---|
| Готовый скор организаторов `attrs+title_score`, лучший порог | 0.697 |
| + правило «у оффера не больше одного матча» | 0.748 |
| LightGBM на признаках пар (без утечки) | 0.787 (holdout) |
| Нейросеть-переранжировщик (listwise, Transformer по кандидатам) | 0.788 (holdout) |
| Смесь LightGBM и сети | 0.795 (Kaggle) |
| **Стекинг: предсказания сети — признаки для LightGBM** | **0.798 (Kaggle)** |

**Как получен итоговый сабмит.** Итоговую модель (последняя строка) подобрал мой AutoML-агент: гонка моделей
LightGBM / HGB, автопризнаки, шаги LLM, подбор порога, обучение на всех данных. Победила обычная LightGBM с параметрами по умолчанию из шаблона агента —
ни автопризнаки, ни шаги LLM не улучшили результат. Этот ноутбук воспроизводит ту же модель без агента:
признаки → нейросеть → LightGBM на тех же фолдах → порог → сабмит.

**План ноутбука**
1. Данные и главные факты.
2. Аудит эмбеддингов: векторы названий товаров не привязаны к id; утечка через наличие фото товара.
3. Разбиение: 5 фолдов и holdout — целыми офферами (офферы test в train не встречаются).
4. Признаки пар для LightGBM.
5. Нейросеть-переранжировщик (PyTorch, GPU): OOF-предсказания `nn_p`, `nn_none`.
6. LightGBM со стекингом, порог по OOF, проверка на holdout.
7. Обучение на всех данных и сабмит.

Время: около часа на GPU P100/T4 (нейросеть — минуты, LightGBM на 2.5 млн строк — основная часть).
Для быстрой проверки: `SAMPLE_OFFERS = 20000`.

### Утечка через наличие фото товара — и почему решение построено без неё

Первая версия признаков дала неожиданный скачок: OOF F1 вырос с **0.80 до 0.945**. Разбор показал, что почти весь
прирост даёт один признак — `has_goods_image`, «у товара-кандидата есть вектор фото в файле»:

| | train |
|---|---|
| фото товара есть у матчей | 99.92% |
| фото товара есть у не-матчей | 11.3% |
| вероятность матча, если фото нет | 0.01% (227 из 292 тыс. матчей) |
| AUC одного этого флага | 0.943 |

Судя по всему, векторы фото при сборке датасета выгружали в основном для товаров, которые где-то являются матчем.
В test устроено так же: доля пар с фото 21.54% при ожидаемых по train 21.67%, а треть товаров с фото в test в train
не встречалась — значит, это свойство всего датасета, а не следствие ответов train. На лидерборде утечка работает:
соревновательная версия набрала **0.946** (private).

**Почему итоговое решение — без неё.** Признак взят из выданных файлов и ответов не содержит, но о сходстве товаров
он ничего не говорит: в реальном маркетплейсе фото есть у карточки независимо от того, подходит ли она продавцу.
Модель, которая опирается на этот флаг, в продакшене бесполезна. Поэтому дальше решение строится **без всего, что
выдаёт наличие фото товара** (`GOODS_IMAGE = False`): нет флага, нет косинуса фото товара и признаков на его основе
(пустой косинус тоже выдавал бы наличие фото). Фото продавца остаётся — его наличие ответа не выдаёт (AUC 0.50).

| Версия | Kaggle private |
|---|---|
| с утечкой (соревновательная) | 0.946 |
| **без утечки (это решение)** | **0.798** |

Цифра ниже, зато она отражает то, насколько модель действительно умеет сопоставлять товары.

In [1]:
import glob
import math
import os
import time
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

SEED = 42
N_FOLDS = 5
HOLDOUT_FRAC = 0.15    # доля офферов train, которых не увидит ни одна модель: честная проверка всего стекинга
GOODS_IMAGE = False    # False — честная версия: без признаков фото товара (их наличие выдаёт ответ, раздел 2)
SAMPLE_OFFERS = 0      # > 0 — быстрая проба на части офферов train (например 20000); 0 — все
NN_EPOCHS = 20         # потолок эпох нейросети (ранняя остановка обычно на 6–10)
REFIT_SEEDS = 1        # агент усреднял 3 сида при обучении на всех данных; 1 — втрое быстрее


def find_data_dir() -> Path:
    for root in ["/kaggle/input", "benchmarks/megamarket/raw", "../benchmarks/megamarket/raw", "."]:
        for hit in sorted(glob.glob(os.path.join(root, "**", "train.csv"), recursive=True)):
            folder = Path(hit).parent
            if (folder / "test.csv").exists():
                return folder
    raise FileNotFoundError("не нашёл train.csv и test.csv: подключите данные конкурса к ноутбуку")


def vec_dir(name: str) -> Path:
    """Папка с векторами (в ней или в её подпапке с тем же именем — embed_deperson.npy и items_deperson.npy)."""
    for root in [RAW, RAW.parent, Path("/kaggle/input")]:
        for hit in sorted(glob.glob(os.path.join(root, "**", name, "**", "embed_deperson.npy"), recursive=True)):
            path = Path(hit).parent
            return path.parent if path.name == name and path.parent.name == name else path
    raise FileNotFoundError(f"не нашёл векторы {name}")


RAW = find_data_dir()
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("notebook_out")
OUT.mkdir(exist_ok=True)
print("данные:", RAW, "| результаты:", OUT)

данные: /kaggle/input/competitions/intern-binary-classification-marketplace-offers | результаты: /kaggle/working


## 1. Данные

Строка train/test — пара «оффер × кандидат». Колонки: `attrs+title_score` — вероятность матча от модели
организаторов (она видела настоящие тексты), цены, длина названий, категория товара. Самих текстов и картинок нет —
только векторы: названия (64 числа) и фото (256 чисел), отдельно для офферов и для товаров.

In [2]:
OFFER, GOODS, SCORE = "offer_depersanalised", "goods_depersanalised", "attrs+title_score"
COLS = [OFFER, GOODS, "sum_length", SCORE, "offer_price", "goods_price", "goods_category_id"]
train = pd.read_csv(RAW / "train.csv", usecols=COLS + ["target", "id"])
test = pd.read_csv(RAW / "test.csv", usecols=COLS + ["id"])
y = train["target"].to_numpy()

per_offer = train.groupby(OFFER)["target"].sum()
distinct = train[train.target == 1].groupby(OFFER)[GOODS].nunique()
print(f"train: {len(train):,} пар, {train[OFFER].nunique():,} офферов; test: {len(test):,} пар, "
      f"{test[OFFER].nunique():,} офферов; общих офферов: {len(set(train[OFFER]) & set(test[OFFER]))}")
print(f"доля матчей {y.mean():.3f}; офферов без матча {(per_offer == 0).mean():.1%}; "
      f"офферов с двумя РАЗНЫМИ товарами-матчами: {(distinct > 1).sum()}")
print(f"повторяющихся id пар в train: {train['id'].duplicated().sum():,} (товар в двух категориях; в test — "
      f"{test['id'].duplicated().sum()})")

train: 2,518,441 пар, 500,000 офферов; test: 363,835 пар, 72,767 офферов; общих офферов: 0
доля матчей 0.117; офферов без матча 41.6%; офферов с двумя РАЗНЫМИ товарами-матчами: 0
повторяющихся id пар в train: 18,441 (товар в двух категориях; в test — 0)


Главное: офферы test в train не встречаются — проверять модели можно только на **новых офферах** (фолды целыми
офферами). У оффера никогда нет двух разных товаров-матчей, а у ~42% офферов матча нет вовсе.

## 2. Аудит эмбеддингов: что в векторах на самом деле есть

In [3]:
@dataclass
class Table:
    """Векторы одного вида: строка i — объект с id ids[i]; последняя строка — нули («вектора нет»)."""
    rows: pd.Index          # id объекта → номер строки (get_indexer; повторы id — первая строка)
    vectors: np.ndarray     # float16 [n + 1, dim]

    @property
    def missing(self) -> int:
        return len(self.vectors) - 1

    def lookup(self, ids: np.ndarray) -> np.ndarray:
        """Номера строк для id; нет вектора → номер нулевой строки."""
        pos = self.rows.get_indexer(ids)
        return np.where(pos >= 0, pos, self.missing).astype(np.int64)


def load_table(folder: Path, how: str) -> Table:
    """how: "standardize" — названия (вычесть среднее, поделить на разброс по каждой координате);
    "unit" — фото (длина вектора → sqrt(dim): тогда косинус = сумма произведений / dim, координаты порядка 1)."""
    sub = next(p for p in [folder / folder.name, folder] if (p / "embed_deperson.npy").exists())
    vec = np.load(sub / "embed_deperson.npy")
    ids = np.load(sub / "items_deperson.npy", allow_pickle=True).astype(str).astype(np.int64)
    first = ~pd.Index(ids).duplicated()
    if not first.all():
        vec, ids = vec[first], ids[first]
    vec = vec.astype(np.float32, copy=False)  # дальше — «на месте», без лишних копий (векторы фото — сотни МБ)
    if how == "standardize":
        vec -= vec.mean(0)
        vec /= vec.std(0) + 1e-6
    else:
        norm = np.sqrt(np.einsum("ij,ij->i", vec, vec))[:, None]
        vec /= np.where(norm > 0, norm, 1.0)
        vec *= math.sqrt(vec.shape[1])
    out = np.zeros((len(vec) + 1, vec.shape[1]), np.float16)
    out[:-1] = vec
    return Table(pd.Index(ids), out)

In [4]:
tables = {"offer_title": load_table(vec_dir("offer_title_vectors"), "standardize"),
          "goods_title": load_table(vec_dir("goods_title_vectors"), "standardize"),
          "offer_image": load_table(vec_dir("offer_image_vectors"), "unit"),
          "goods_image": load_table(vec_dir("goods_image_vectors"), "unit")}


def auc(score: np.ndarray, target: np.ndarray) -> float:
    r = pd.Series(score).rank().to_numpy()
    pos = target.sum()
    return float((r[target == 1].sum() - pos * (pos + 1) / 2) / (pos * (len(target) - pos)))


rng = np.random.default_rng(0)
offers_all = train[OFFER].unique()
sample = train[train[OFFER].isin(rng.choice(offers_all, min(60000, len(offers_all)), replace=False))]
ys = sample["target"].to_numpy()


def pair_cos(kind: str) -> tuple[np.ndarray, np.ndarray]:
    o, g = tables[f"offer_{kind}"], tables[f"goods_{kind}"]
    oi, gi = o.lookup(sample[OFFER].to_numpy()), g.lookup(sample[GOODS].to_numpy())
    ok = (oi != o.missing) & (gi != g.missing)
    a, b = o.vectors[oi[ok]].astype(np.float32), g.vectors[gi[ok]].astype(np.float32)
    cos = (a * b).sum(1) / np.linalg.norm(a, axis=1) / np.linalg.norm(b, axis=1)
    return cos, ok


for kind in ("title", "image"):
    cos, ok = pair_cos(kind)
    print(f"косинус {kind}: есть у {ok.mean():.1%} пар, AUC {auc(cos, ys[ok]):.3f}")

has_img = (tables["goods_image"].lookup(train[GOODS].to_numpy()) != tables["goods_image"].missing)
has_img_test = (tables["goods_image"].lookup(test[GOODS].to_numpy()) != tables["goods_image"].missing)
print(f"\nвектор фото товара есть: у матчей {has_img[y == 1].mean():.2%}, у не-матчей {has_img[y == 0].mean():.2%}; "
      f"P(матч | фото нет) = {y[~has_img].mean():.4f}; AUC самого флага {auc(has_img.astype(float), y):.3f}")
print(f"доля пар с фото товара: train {has_img.mean():.2%}, test {has_img_test.mean():.2%}")
del tables["goods_title"]

косинус title: есть у 100.0% пар, AUC 0.504
косинус image: есть у 17.5% пар, AUC 0.742

вектор фото товара есть: у матчей 99.92%, у не-матчей 11.30%; P(матч | фото нет) = 0.0001; AUC самого флага 0.943
доля пар с фото товара: train 21.67%, test 21.54%


**Выводы аудита** (на полном train: те же цифры)

* **Векторы названий товаров к id привязаны неверно**: косинус «название оффера ↔ название товара» — AUC 0.50
  (монетка). Обученное отображение одного пространства в другое тоже не находит связи (R² = 0.00), хотя у самих
  векторов структура есть (у офферов с одним товаром-матчем косинус названий 0.956). Признаки из них — шум.
* **Векторы фото работают** (AUC ~0.74), но оба фото есть лишь у ~17.5% пар.
* **Утечка через наличие фото товара**: вектор фото выгружен почти только для товаров-матчей (99.9% матчей против
  11% не-матчей), одно наличие фото даёт AUC 0.94. В test так же (доля пар с фото та же), поэтому на лидерборде это
  «работает» (соревновательная версия — 0.946), но о сходстве товаров ничего не говорит. **Дальше — честная
  версия: без фото товара** (`GOODS_IMAGE = False`); фото продавца остаётся.

## 3. Разбиение: фолды и holdout — целыми офферами

Одно разбиение на всех: 15% офферов — holdout (его не видит ни нейросеть, ни LightGBM), остальные — 5 фолдов.
Нейросеть и LightGBM используют одни и те же фолды: тогда `nn_p` у каждой строки train сделан моделью, которая эту
строку не видела, и как признак не подсказывает ответ.

In [5]:
def make_split(offer_ids: np.ndarray, n_folds: int, seed: int, holdout_frac: float = 0.0) -> np.ndarray:
    """Фолд каждого оффера, −1 — holdout (доля holdout_frac офферов, которых не увидит ни одна модель сети).
    holdout_frac = 0 — то же, что make_folds."""
    folds = np.empty(len(offer_ids), np.int64)
    n_hold = int(round(holdout_frac * len(offer_ids)))
    order = np.random.default_rng(seed + 1000).permutation(len(offer_ids))
    held, rest = order[:n_hold], np.sort(order[n_hold:])
    folds[held] = -1
    folds[rest] = make_folds(offer_ids[rest], n_folds, seed)
    return folds


def make_folds(offer_ids: np.ndarray, n_folds: int, seed: int) -> np.ndarray:
    """Фолд каждого оффера: случайная перестановка, затем по кругу 0, 1, …, n_folds − 1 (размеры почти равны)."""
    order = np.random.default_rng(seed).permutation(len(offer_ids))
    folds = np.empty(len(offer_ids), np.int64)
    folds[order] = np.arange(len(offer_ids)) % n_folds
    return folds

## 4. Признаки пар для LightGBM

Как в конвертере проекта (`bench/megamarket_features.py`): косинусы векторов, цены, популярность товара и признаки
«внутри оффера» — место кандидата, отрыв от лучшего и от остальных (по скору, косинусу названий, косинусу фото).
`title_cos` — шум (раздел 2), но оставлен: итоговая модель агента училась с ним, ноутбук повторяет её точно.

In [6]:
CHUNK = 200_000  # пар за раз: 200 тыс. × 256 чисел × 4 байта ≈ 200 МБ на сторону


def load_vectors(folder: Path) -> tuple[dict, np.ndarray]:
    """(id строкой → номер строки, векторы единичной длины float32)."""
    sub = next(p for p in [folder / folder.name, folder] if (p / "embed_deperson.npy").exists())
    vec = np.load(sub / "embed_deperson.npy").astype(np.float32)
    ids = np.load(sub / "items_deperson.npy", allow_pickle=True).astype(str)
    norm = np.linalg.norm(vec, axis=1, keepdims=True)
    vec /= np.where(norm > 0, norm, 1.0)
    return {k: i for i, k in enumerate(ids)}, vec


def pair_cosine(left_ids: pd.Series, right_ids: pd.Series, left: tuple, right: tuple) -> np.ndarray:
    """Косинус векторов пары; NaN, если у одной из сторон вектора нет."""
    (lmap, lvec), (rmap, rvec) = left, right
    li = left_ids.astype(str).map(lmap).fillna(-1).astype(np.int64).to_numpy()
    ri = right_ids.astype(str).map(rmap).fillna(-1).astype(np.int64).to_numpy()
    out = np.full(len(li), np.nan, dtype=np.float32)
    ok = np.flatnonzero((li >= 0) & (ri >= 0))
    for start in range(0, len(ok), CHUNK):
        part = ok[start:start + CHUNK]
        out[part] = np.einsum("ij,ij->i", lvec[li[part]], rvec[ri[part]])
    return out


def group_features(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Внутри оффера: место (1 — наибольшее значение), отрыв от лучшего и от среднего остальных кандидатов."""
    g = df.groupby(OFFER, sort=False)
    out = pd.DataFrame(index=df.index)
    out["offer_candidates"] = g[OFFER].transform("size").astype(np.int16)
    for c in cols:
        out[f"{c}_rank"] = g[c].rank(ascending=False, method="min").astype(np.float32)
        top = g[c].transform("max")
        out[f"{c}_gap_top"] = (top - df[c]).astype(np.float32)
        total, count = g[c].transform("sum"), g[c].transform("count")
        others = (total - df[c].fillna(0)) / (count - df[c].notna().astype(int)).replace(0, np.nan)
        out[f"{c}_vs_others"] = (df[c] - others).astype(np.float32)
    return out

In [7]:
def pair_table(train: pd.DataFrame, test: pd.DataFrame, goods_image: bool) -> pd.DataFrame:
    """Признаки пар train + test подряд (как build() конвертера, без записи файлов)."""
    both = pd.concat([train.drop(columns=["target"]), test], ignore_index=True)
    for kind in ("title", "image"):
        offer = load_vectors(vec_dir(f"offer_{kind}_vectors"))
        if kind == "image" and not goods_image:
            both["has_offer_image"] = both[OFFER].astype(str).isin(offer[0]).astype(np.int8)
            continue
        goods = load_vectors(vec_dir(f"goods_{kind}_vectors"))
        both[f"{kind}_cos"] = pair_cosine(both[OFFER], both[GOODS], offer, goods)
        if kind == "image":
            both["has_offer_image"] = both[OFFER].astype(str).isin(offer[0]).astype(np.int8)
            both["has_goods_image"] = both[GOODS].astype(str).isin(goods[0]).astype(np.int8)
        del offer, goods
    price = both["goods_price"].where(both["goods_price"] > 0)
    both["price_ratio"] = (both["offer_price"] / price).astype(np.float32)
    both["price_log_diff"] = (np.log1p(both["offer_price"].clip(lower=0)) - np.log1p(price)).astype(np.float32)
    both["goods_pairs"] = both.groupby(GOODS)[GOODS].transform("size").astype(np.int32)
    group_cols = [SCORE, "title_cos"] + (["image_cos"] if goods_image else [])
    both = pd.concat([both, group_features(both, group_cols)], axis=1)
    both["offer_group__freq"] = both.groupby(OFFER)[OFFER].transform("size").astype(float)  # как у агента
    both["goods_category_id"] = "c" + both["goods_category_id"].astype("Int64").astype(str)
    return both.drop(columns=[OFFER, GOODS, "id"])


started = time.time()
pairs = pair_table(train, test, GOODS_IMAGE)
print(f"признаков пар: {pairs.shape[1]} за {time.time() - started:.0f} с: {list(pairs.columns)}")

признаков пар: 18 за 16 с: ['sum_length', 'attrs+title_score', 'offer_price', 'goods_price', 'goods_category_id', 'title_cos', 'has_offer_image', 'price_ratio', 'price_log_diff', 'goods_pairs', 'offer_candidates', 'attrs+title_score_rank', 'attrs+title_score_gap_top', 'attrs+title_score_vs_others', 'title_cos_rank', 'title_cos_gap_top', 'title_cos_vs_others', 'offer_group__freq']


## 5. Нейросеть-переранжировщик

LightGBM смотрит на каждую пару отдельно. Сеть смотрит на оффер и **сразу на всех его кандидатов** и выбирает один
вариант: «кандидат 1…K или никто» (softmax). «Не больше одного матча на оффер» встроено в модель, а вариант «никто»
учит распознавать офферы без матча.

* вход: название оффера (64), фото продавца и фото кандидатов (256, общая проекция), категория и 14 числовых
  признаков кандидата (скор, место, отрыв от лучшего, позиция в выдаче, цены, популярность товара);
* взаимодействия: фото × фото, название × фото, название × категория (поэлементные произведения) + косинус фото;
* Transformer-энкодер по кандидатам (2 слоя × 4 головы, d = 128): кандидаты «смотрят» друг на друга;
* потери — перекрёстная энтропия по правильному варианту (listwise); 689 тыс. параметров.

Готовой языковой модели здесь не на чем обучать: текстов в данных нет, только векторы.

In [8]:
FEATURES = ["score", "score_logit", "score_rank", "score_gap_top", "position", "n_candidates", "log_sum_length",
            "has_goods_price", "log_goods_price", "log_price_diff", "log_offer_price", "log_goods_pairs",
            "has_goods_image", "has_offer_image"]


@dataclass
class Offers:
    """Данные по офферам (train и test вместе): оффер — строка, кандидаты — до K мест; пустые места — mask False."""
    offer_ids: np.ndarray   # [N] id оффера
    is_test: np.ndarray     # [N] bool
    title: np.ndarray       # [N] строка в таблице названий офферов
    offer_image: np.ndarray  # [N] строка в таблице фото офферов (нет фото → нулевая строка)
    goods_image: np.ndarray  # [N, K] строка в таблице фото товаров
    category: np.ndarray    # [N, K] 0 — пустое место, 1… — категории
    num: np.ndarray         # [N, K, F] числовые признаки (стандартизованы)
    mask: np.ndarray        # [N, K] есть ли кандидат на месте
    y: np.ndarray           # [N, K] таргет (у test — нули)
    row: np.ndarray         # [N, K] номер строки в исходном train.csv / test.csv (пустое место — −1)
    goods: np.ndarray       # [N, K] id товара (для правила top-1; пустое место — −1)
    n_categories: int
    feature_mean: np.ndarray
    feature_std: np.ndarray


def pair_features(df: pd.DataFrame, goods_pairs: pd.Series, has_goods_img: np.ndarray,
                  has_offer_img: np.ndarray) -> np.ndarray:
    """Числовые признаки каждой пары [строк, F] в порядке FEATURES (до стандартизации)."""
    g = df.groupby(OFFER, sort=False)[SCORE]
    score = df[SCORE].astype(float).fillna(0.0)
    clipped = score.clip(1e-4, 1 - 1e-4)
    goods_price = df["goods_price"].where(df["goods_price"] > 0)
    log_gp = np.log1p(goods_price)
    log_op = np.log1p(df["offer_price"].clip(lower=0).fillna(0))
    cols = {
        "score": score,
        "score_logit": np.log(clipped / (1 - clipped)),
        "score_rank": g.rank(ascending=False, method="min").fillna(5),
        "score_gap_top": (g.transform("max") - score).fillna(0),
        "position": df.groupby(OFFER, sort=False).cumcount(),  # порядок поисковой выдачи (в train и test одинаков)
        "n_candidates": df.groupby(OFFER, sort=False)[OFFER].transform("size"),
        "log_sum_length": np.log1p(df["sum_length"].clip(lower=0).fillna(0)),
        "has_goods_price": goods_price.notna().astype(float),
        "log_goods_price": log_gp.fillna(0),
        "log_price_diff": (log_op - log_gp).fillna(0),
        "log_offer_price": log_op,
        "log_goods_pairs": np.log1p(df[GOODS].map(goods_pairs).fillna(0)),
        "has_goods_image": has_goods_img.astype(float),
        "has_offer_image": has_offer_img.astype(float),
    }
    return np.column_stack([np.asarray(cols[f], dtype=np.float32) for f in FEATURES])


def prepare(train: pd.DataFrame, test: pd.DataFrame, tables: dict[str, Table], goods_image: bool = True) -> Offers:
    """Пары из CSV → массивы по офферам. Строки оффера идут в порядке файла (это и есть «позиция»).
    goods_image=False — у всех кандидатов «фото нет» (честная версия, без утечки через наличие фото)."""
    train = train.assign(_part=0, _row=np.arange(len(train)))
    test = test.assign(_part=1, _row=np.arange(len(test)), target=0)
    both = pd.concat([train, test], ignore_index=True)
    goods_pairs = both.groupby(GOODS).size()
    gimg = tables["goods_image"].lookup(both[GOODS].to_numpy())
    if not goods_image:
        gimg[:] = tables["goods_image"].missing
    oimg = tables["offer_image"].lookup(both[OFFER].to_numpy())
    feats = pair_features(both, goods_pairs, gimg != tables["goods_image"].missing,
                          oimg != tables["offer_image"].missing)

    # оффер → номер строки N, кандидат → место k (порядок в файле)
    offer_code, offer_ids = pd.factorize(both[OFFER], sort=False)
    slot = both.groupby(OFFER, sort=False).cumcount().to_numpy()
    n, k = len(offer_ids), int(slot.max()) + 1
    cats = both["goods_category_id"].astype("float64")
    cat_values = np.sort(cats.dropna().unique())
    cat_code = np.where(cats.isna(), len(cat_values) + 1, np.searchsorted(cat_values, cats.fillna(-1)) + 1)

    def grid(values, fill, dtype):
        out = np.full((n, k) + np.shape(values)[1:], fill, dtype=dtype)
        out[offer_code, slot] = values
        return out

    mask = grid(np.ones(len(both), bool), False, bool)
    num = grid(feats, 0.0, np.float32)
    real = mask & (grid(both["_part"].to_numpy(), 1, np.int8) == 0)  # стандартизация — по кандидатам train
    values = num[real].astype(np.float64)  # float64: у признака-константы разброс ровно 0, а не шум округления
    mean, std = values.mean(0), values.std(0)
    std = np.where(std > 1e-9, std, 1.0)    # константа → после вычитания среднего нули
    num -= mean.astype(np.float32)          # на месте: массив — сотни МБ
    num /= std.astype(np.float32)
    num[~mask] = 0.0
    first = np.unique(offer_code, return_index=True)[1]  # первая строка каждого оффера (коды — по порядку появления)
    return Offers(
        offer_ids=np.asarray(offer_ids), is_test=both["_part"].to_numpy()[first] == 1,
        title=tables["offer_title"].lookup(np.asarray(offer_ids)), offer_image=oimg[first],
        goods_image=grid(gimg, tables["goods_image"].missing, np.int64), category=grid(cat_code, 0, np.int64),
        num=num, mask=mask, y=grid(both["target"].to_numpy(dtype=np.float32), 0.0, np.float32),
        row=grid(both["_row"].to_numpy(), -1, np.int64), goods=grid(both[GOODS].to_numpy(), -1, np.int64),
        n_categories=len(cat_values) + 2, feature_mean=mean, feature_std=std)

In [9]:
def build_model(n_num: int, n_categories: int, d: int, layers: int, heads: int, dropout: float):
    import torch
    from torch import nn

    class Matcher(nn.Module):
        """Оффер + K кандидатов → K + 1 чисел (кандидаты и «никто»); softmax по ним — вероятности вариантов."""

        def __init__(self):
            super().__init__()
            self.title = nn.Sequential(nn.Linear(64, d), nn.GELU(), nn.Linear(d, d))             # ① название оффера
            self.image = nn.Sequential(nn.Linear(256, d), nn.LayerNorm(d))                     # ② фото: общая проекция
            self.category = nn.Embedding(n_categories, d, padding_idx=0)
            self.numeric = nn.Sequential(nn.Linear(n_num + 1, d), nn.GELU(), nn.Linear(d, d))  # +1 — косинус фото
            self.candidate = nn.Sequential(nn.Linear(8 * d, 2 * d), nn.GELU(), nn.Dropout(dropout),
                                           nn.Linear(2 * d, d))                                # ④ вектор кандидата
            self.none = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Linear(d, d))          # ④ токен «никто»
            layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout, activation="gelu", batch_first=True,
                                               norm_first=True)
            self.context = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)     # ⑤ внимание
            self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1))                        # ⑥ оценка

        def forward(self, title, offer_img, has_offer_img, goods_img, has_goods_img, category, num, mask):
            # title [B,64]; offer_img [B,256]; has_offer_img [B]; goods_img [B,K,256]; has_goods_img [B,K];
            # category [B,K]; num [B,K,F]; mask [B,K] — True, если кандидат на месте есть
            t = self.title(title)                                        # [B,d]
            o = self.image(offer_img) * has_offer_img[:, None]           # [B,d]; нет фото → нули
            g = self.image(goods_img) * has_goods_img[..., None]         # [B,K,d]
            c = self.category(category)                                  # [B,K,d]
            both = has_offer_img[:, None] * has_goods_img                # фото есть с обеих сторон
            cos = (offer_img[:, None, :] * goods_img).sum(-1) / offer_img.shape[-1] * both
            x = self.numeric(torch.cat([num, cos[..., None]], -1))       # [B,K,d]
            te, oe = t[:, None, :].expand_as(g), o[:, None, :].expand_as(g)
            pair = torch.cat([te, oe, g, c, x, oe * g, te * g, te * c], -1)  # ③ взаимодействия — произведения
            tokens = torch.cat([self.candidate(pair), self.none(torch.cat([t, o], -1))[:, None, :]], 1)
            skip = torch.cat([~mask, torch.zeros_like(mask[:, :1])], 1)   # пустые места внимание пропускает
            tokens = self.context(tokens, src_key_padding_mask=skip)
            return self.head(tokens).squeeze(-1).float().masked_fill(skip, float("-inf"))  # [B,K+1]

    return Matcher()


def listwise_loss(logits, y):
    """Перекрёстная энтропия по правильному варианту: матч → его место (повтор товара — поровну между строками),
    матча нет → «никто». logits [B,K+1] (пустые места = −inf), y [B,K]."""
    import torch
    found = y.sum(1, keepdim=True)
    target = torch.cat([y / found.clamp(min=1), (found == 0).float()], 1)
    logp = torch.log_softmax(logits, 1)
    return -torch.where(target > 0, target * logp, torch.zeros_like(logp)).sum(1).mean()


class Trainer:
    """Все массивы — на видеокарте; батч — номера офферов, векторы достаются из таблиц по номерам строк."""

    def __init__(self, data: Offers, tables: dict[str, Table], args, device: str):
        import torch
        self.torch, self.args, self.device = torch, args, device
        t = lambda a, dtype=None: torch.as_tensor(a, device=device, dtype=dtype)  # noqa: E731
        self.tab = {name: t(tab.vectors) for name, tab in tables.items()}
        self.missing = {name: tab.missing for name, tab in tables.items()}
        self.d = {"title": t(data.title), "offer_image": t(data.offer_image), "goods_image": t(data.goods_image),
                  "category": t(data.category), "num": t(data.num), "mask": t(data.mask), "y": t(data.y)}
        self.n_categories = data.n_categories
        self.amp = device.startswith("cuda")

    def batch(self, idx):
        d, tab = self.d, self.tab
        oi, gi = d["offer_image"][idx], d["goods_image"][idx]
        return (tab["offer_title"][d["title"][idx]].float(), tab["offer_image"][oi].float(),
                (oi != self.missing["offer_image"]).float(), tab["goods_image"][gi].float(),
                (gi != self.missing["goods_image"]).float(), d["category"][idx], d["num"][idx], d["mask"][idx])

    def logits(self, model, idx):
        with self.torch.autocast("cuda", dtype=self.torch.bfloat16, enabled=self.amp):
            return model(*self.batch(idx))

    def mean_loss(self, model, idx) -> float:
        torch = self.torch
        model.eval()
        total = 0.0
        with torch.no_grad():
            for s in range(0, len(idx), self.args.batch * 4):
                b = idx[s:s + self.args.batch * 4]
                total += listwise_loss(self.logits(model, b), self.d["y"][b]).item() * len(b)
        return total / max(len(idx), 1)

    def predict(self, model, idx) -> np.ndarray:
        torch = self.torch
        model.eval()
        out = []
        with torch.no_grad():
            for s in range(0, len(idx), self.args.batch * 4):
                out.append(torch.softmax(self.logits(model, idx[s:s + self.args.batch * 4]), 1).cpu().numpy())
        return np.concatenate(out) if out else np.zeros((0, self.d["mask"].shape[1] + 1), np.float32)

    def fit(self, train_idx: np.ndarray, valid_idx: np.ndarray, log) -> tuple[object, dict]:
        torch, a = self.torch, self.args
        model = build_model(len(FEATURES), self.n_categories, a.d, a.layers, a.heads, a.dropout).to(self.device)
        train_idx, valid_idx = torch.as_tensor(train_idx, device=self.device), torch.as_tensor(valid_idx, device=self.device)
        opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.weight_decay)
        total = max(math.ceil(len(train_idx) / a.batch) * a.max_epochs, 4)
        # разогрев — 5% шагов, но не меньше 2 (25.10: на пробе 20 шагов → 0 шагов разогрева → деление на ноль в torch)
        warmup = min(0.5, max(0.05, 2 / total))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=total, pct_start=warmup)
        best, best_state, bad, history = math.inf, None, 0, []
        for epoch in range(a.max_epochs):
            started = time.monotonic()
            model.train()
            order = train_idx[torch.randperm(len(train_idx), device=self.device)]
            total = 0.0
            for s in range(0, len(order), a.batch):
                b = order[s:s + a.batch]
                loss = listwise_loss(self.logits(model, b), self.d["y"][b])
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                sched.step()
                total += loss.item() * len(b)
            valid = self.mean_loss(model, valid_idx)
            history.append({"epoch": epoch + 1, "train_loss": round(total / len(order), 5), "valid_loss": round(valid, 5),
                            "seconds": round(time.monotonic() - started, 1)})
            log(f"    эпоха {epoch + 1:2d}: потери train {total / len(order):.4f}, valid {valid:.4f} "
                f"({time.monotonic() - started:.0f} с)")
            if valid < best - 1e-4:
                best, bad = valid, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= a.patience:
                    break
        if best_state is not None:  # None — потери ни разу не были числом (NaN): остаётся модель последней эпохи
            model.load_state_dict(best_state)
        return model, {"best_valid_loss": round(best, 5), "epochs": history}

In [10]:
import torch

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
args = SimpleNamespace(batch=1024, lr=2e-3, weight_decay=1e-4, max_epochs=NN_EPOCHS, patience=3, d=128, layers=2,
                       heads=4, dropout=0.1)
data = prepare(train[COLS + ["target"]], test[COLS], tables, goods_image=GOODS_IMAGE)
train_offers, test_offers = np.flatnonzero(~data.is_test), np.flatnonzero(data.is_test)
split = make_split(data.offer_ids[train_offers], N_FOLDS, SEED, HOLDOUT_FRAC)   # фолд 0…4, −1 — holdout
use = np.ones(len(train_offers), bool)
if SAMPLE_OFFERS:
    use[:] = False
    use[np.random.default_rng(SEED).choice(len(train_offers), SAMPLE_OFFERS, replace=False)] = True
held = train_offers[use & (split < 0)]
width = data.mask.shape[1] + 1
oof = np.full((len(data.offer_ids), width), np.nan, np.float32)
test_prob = np.zeros((len(test_offers), width), np.float32)
held_prob = np.zeros((len(held), width), np.float32)
# test и holdout — предсказание ОДНОЙ (случайной) модели фолда, как OOF у train: среднее пяти моделей «мягче»,
# и LightGBM видел бы в nn_p сдвиг train/test (детектор сдвига: ROC-AUC 0.60 против 0.51)
pick = np.random.default_rng(SEED + 2000)
test_model, held_model = pick.permutation(len(test_offers)) % N_FOLDS, pick.permutation(len(held)) % N_FOLDS

trainer = Trainer(data, tables, args, device)
trainer.amp = device == "cuda" and torch.cuda.is_bf16_supported()  # bf16 есть не на всех GPU (T4, P100 — нет)
print(f"устройство {device}, bf16: {trainer.amp}; офферов train {len(train_offers):,}, test {len(test_offers):,}")
started = time.time()
for f in range(N_FOLDS):
    fit = train_offers[use & (split >= 0) & (split != f)]
    valid_mask = np.random.default_rng(SEED + f).random(len(fit)) < 0.05   # ранняя остановка
    valid_mask[0] = True
    print(f"фолд {f + 1}/{N_FOLDS}: учим на {(~valid_mask).sum():,} офферах")
    model, info = trainer.fit(fit[~valid_mask], fit[valid_mask], print)
    own = train_offers[use & (split == f)]
    oof[own] = trainer.predict(model, torch.as_tensor(own, device=device))
    for offers, prob, owner in ((test_offers, test_prob, test_model), (held, held_prob, held_model)):
        mine = owner == f
        prob[mine] = trainer.predict(model, torch.as_tensor(offers[mine], device=device))
    del model
oof[held] = held_prob
print(f"нейросеть: {time.time() - started:.0f} с")


def to_rows(offers: np.ndarray, prob: np.ndarray, n_rows: int) -> tuple[np.ndarray, np.ndarray]:
    """Вероятности по офферам → по строкам исходного CSV: nn_p (кандидат — матч), nn_none (матча нет)."""
    mask = data.mask[offers]
    rows = data.row[offers][mask]
    p, none = np.full(n_rows, np.nan, np.float32), np.full(n_rows, np.nan, np.float32)
    p[rows] = prob[:, :-1][mask]
    none[rows] = np.repeat(prob[:, -1], mask.sum(1))
    return p, none


used = train_offers[use]
nn_p_train, nn_none_train = to_rows(used, oof[used], len(train))
nn_p_test, nn_none_test = to_rows(test_offers, test_prob, len(test))
fold_of_offer = pd.Series(split, index=data.offer_ids[train_offers])
cv_fold = train[OFFER].map(fold_of_offer).to_numpy()   # фолд каждой строки train, −1 — holdout
in_use = ~np.isnan(nn_p_train)

устройство cuda, bf16: True; офферов train 500,000, test 72,767
фолд 1/5: учим на 323,141 офферах
    эпоха  1: потери train 0.6490, valid 0.5173 (13 с)
    эпоха  2: потери train 0.5062, valid 0.5003 (12 с)
    эпоха  3: потери train 0.4895, valid 0.4933 (12 с)
    эпоха  4: потери train 0.4776, valid 0.4903 (12 с)
    эпоха  5: потери train 0.4663, valid 0.4872 (12 с)
    эпоха  6: потери train 0.4545, valid 0.4886 (12 с)
    эпоха  7: потери train 0.4416, valid 0.4938 (13 с)
    эпоха  8: потери train 0.4280, valid 0.4965 (13 с)
фолд 2/5: учим на 323,123 офферах
    эпоха  1: потери train 0.6339, valid 0.4970 (13 с)
    эпоха  2: потери train 0.5079, valid 0.4845 (13 с)
    эпоха  3: потери train 0.4908, valid 0.4777 (13 с)
    эпоха  4: потери train 0.4783, valid 0.4730 (13 с)
    эпоха  5: потери train 0.4670, valid 0.4732 (13 с)
    эпоха  6: потери train 0.4550, valid 0.4662 (13 с)
    эпоха  7: потери train 0.4421, valid 0.4708 (13 с)
    эпоха  8: потери train 0.4284, valid 0.

In [11]:
def f1_score(y: np.ndarray, pred: np.ndarray) -> float:
    """F1 класса 1 по всем парам сразу: 2·TP / (2·TP + FP + FN)."""
    tp = int(np.sum((pred == 1) & (y == 1)))
    fp = int(np.sum((pred == 1) & (y == 0)))
    fn = int(np.sum((pred == 0) & (y == 1)))
    return 2 * tp / (2 * tp + fp + fn) if tp else 0.0


def best_threshold(score: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """Порог под максимум F1 за один проход: сортируем по убыванию, «1» — первые k строк, перебираем все k.

    F1 при первых k строках = 2·TP(k) / (k + P), где TP(k) — матчей среди первых k, P — всего матчей.
    Резать можно только между разными значениями (равные вероятности получают одинаковый ответ).
    Строки с score < 0 (не лучшие кандидаты при top-1) единицу не получают никогда.
    Возвращает (порог, F1); ответ — score >= порог.
    """
    order = np.argsort(-score, kind="stable")
    s, yy = score[order], y[order]
    tp = np.cumsum(yy)
    k = np.arange(1, len(s) + 1)
    f1 = 2 * tp / (k + y.sum())
    can_cut = np.r_[s[:-1] > s[1:], True] & (s >= 0)
    if not can_cut.any():
        return 1.0, 0.0
    best = int(np.argmax(np.where(can_cut, f1, -1)))
    return float(s[best]), float(f1[best])

In [12]:
first = ~train["id"].duplicated().to_numpy()   # повторы пар в оценке — один раз, как в test


def report(name: str, p: np.ndarray, rows_fit: np.ndarray, rows_check: np.ndarray) -> float:
    """Порог — по rows_fit (OOF), F1 — там же и на rows_check (holdout; порог там не подбирается)."""
    fit, check = rows_fit & first, rows_check & first
    thr, f1_fit = best_threshold(p[fit], y[fit])
    f1_check = f1_score(y[check], (p[check] >= thr).astype(int))
    print(f"{name}: OOF F1 {f1_fit:.4f} | holdout F1 {f1_check:.4f} (порог {thr:.3f} — по OOF)")
    return thr


visible, holdout = in_use & (cv_fold >= 0), in_use & (cv_fold < 0)
_ = report("нейросеть, nn_p", np.nan_to_num(nn_p_train), visible, holdout)
_ = report("скор организаторов", train[SCORE].to_numpy(), visible, holdout)

нейросеть, nn_p: OOF F1 0.7862 | holdout F1 0.7877 (порог 0.397 — по OOF)
скор организаторов: OOF F1 0.6964 | holdout F1 0.6979 (порог 0.404 — по OOF)


## 6. LightGBM со стекингом

Признаки пар + `nn_p`, `nn_none`. Параметры — по умолчанию из шаблона агента (их и выбрал агент): шаг 0.03,
15 листьев, до 5000 деревьев с ранней остановкой по 10% обучающей части фолда. Фолды — те же, что у сети.

In [13]:
import lightgbm as lgb

LGB_PARAMS = dict(n_estimators=5000, learning_rate=0.03, num_leaves=15, min_child_samples=20, subsample=0.8,
                  subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbose=-1)
n = len(train)
features = pairs.copy()
features["nn_p"] = np.r_[nn_p_train, nn_p_test]
features["nn_none"] = np.r_[nn_none_train, nn_none_test]
categories = sorted(features["goods_category_id"].iloc[:n].dropna().unique())
features["goods_category_id"] = pd.Categorical(features["goods_category_id"], categories=categories)
X, X_test = features.iloc[:n].reset_index(drop=True), features.iloc[n:].reset_index(drop=True)
print("признаки LightGBM:", list(X.columns))


def fit_lgb(rows: np.ndarray, seed: int):
    """Как fit_model шаблона агента: ранняя остановка по 10% обучающих строк (строки проверки она не видит)."""
    idx = np.flatnonzero(rows)
    inner = np.random.default_rng(seed).random(len(idx)) < 0.10
    model = lgb.LGBMClassifier(random_state=seed, **LGB_PARAMS)
    model.fit(X.iloc[idx[~inner]], y[idx[~inner]], eval_set=[(X.iloc[idx[inner]], y[idx[inner]])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
    return model


oof_lgb = np.full(n, np.nan)
hold_lgb = np.zeros(n)
started = time.time()
for f in range(N_FOLDS):
    model = fit_lgb(visible & (cv_fold != f), SEED)
    own = visible & (cv_fold == f)
    oof_lgb[own] = model.predict_proba(X[own])[:, 1]
    hold_lgb[holdout] += model.predict_proba(X[holdout])[:, 1] / N_FOLDS
    print(f"фолд {f + 1}: деревьев {model.best_iteration_}, {time.time() - started:.0f} с")
p_lgb = np.where(visible, oof_lgb, hold_lgb)
threshold = report("LightGBM + nn_p (стекинг)", np.nan_to_num(p_lgb), visible, holdout)

признаки LightGBM: ['sum_length', 'attrs+title_score', 'offer_price', 'goods_price', 'goods_category_id', 'title_cos', 'has_offer_image', 'price_ratio', 'price_log_diff', 'goods_pairs', 'offer_candidates', 'attrs+title_score_rank', 'attrs+title_score_gap_top', 'attrs+title_score_vs_others', 'title_cos_rank', 'title_cos_gap_top', 'title_cos_vs_others', 'offer_group__freq', 'nn_p', 'nn_none']
фолд 1: деревьев 4995, 500 с
фолд 2: деревьев 4989, 1025 с
фолд 3: деревьев 4975, 1521 с
фолд 4: деревьев 4999, 2052 с
фолд 5: деревьев 5000, 2571 с
LightGBM + nn_p (стекинг): OOF F1 0.7942 | holdout F1 0.7962 (порог 0.408 — по OOF)


## 7. Обучение на всех данных и сабмит

Как у агента: модели фолдов на всём train (вместе с holdout — офферы holdout раздаются по фолдам), предсказания для
test усредняются; порог — подобранный по OOF выше. Агент усреднял 3 сида (`REFIT_SEEDS = 3`).

In [14]:
full_fold = cv_fold.copy()
hold_offers = pd.unique(train.loc[cv_fold < 0, OFFER])
full_fold[cv_fold < 0] = train.loc[cv_fold < 0, OFFER].map(
    pd.Series(np.arange(len(hold_offers)) % N_FOLDS, index=hold_offers)).to_numpy()
test_lgb = np.zeros(len(test))
started = time.time()
for k in range(REFIT_SEEDS):
    for f in range(N_FOLDS):
        model = fit_lgb(in_use & (full_fold != f), SEED + k)
        test_lgb += model.predict_proba(X_test)[:, 1] / (N_FOLDS * REFIT_SEEDS)
    print(f"сид {SEED + k}: {time.time() - started:.0f} с")

sample_sub = pd.read_csv(RAW / "sample_submission.csv", usecols=["id"])
answer = pd.Series((test_lgb >= threshold).astype(int), index=test["id"].to_numpy())
submission = pd.DataFrame({"id": sample_sub["id"], "target": answer.loc[sample_sub["id"]].to_numpy()})
submission.to_csv(OUT / "submission.csv", index=False)
print(f"сабмит: {OUT / 'submission.csv'} — {len(submission):,} строк, единиц {submission.target.mean():.2%}")

сид 42: 2504 с
сабмит: /kaggle/working/submission.csv — 363,835 строк, единиц 12.05%


## Итоги

* **Валидация не врёт:** фолды и holdout — целыми офферами, общие для сети и LightGBM. Порядок четырёх сабмитов по
  holdout совпал с порядком на лидерборде; Kaggle выше holdout на 0.0015–0.003.
* **Что дало качество:** правило «один матч на оффер» над готовым скором (+0.05); признаки «внутри оффера» и
  нейросеть, которая сравнивает кандидатов между собой и с вариантом «никто» (+0.04); стекинг сети с LightGBM (+0.01).
* **Что не дало:** признаки из векторов названий товаров (сломаны), регуляризация сети, автопризнаки и шаги LLM агента.
* **Найдено в данных:** неверная привязка векторов названий товаров к id; утечка через наличие фото товара;
  слабый след сборки в номерах id товаров (AUC 0.57) — id в признаки не берутся.
* **Дальше:** признаки из истории товаров (средний вектор названий офферов, совпавших с товаром; похожие офферы, kNN)
  на тех же фолдах — в пробе +0.005; с сырыми текстами — дообучение кросс-энкодера.